In [ ]:
# importation des bibliotheques
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns
!pip install ydata-profiling
from ydata_profiling import ProfileReport

In [ ]:
# chargement de dataset

df = pd.read_csv('hotel_bookings.csv')

**Nous voulons prédire si une réservation d'hôtel sera annulée (variable cible : is_canceled).**
Objectif :
- Identifier les facteurs qui influencent fortement les annulations
- Aider les hôtels à mieux anticiper les annulations pour optimiser leur revenue management
- Réduire les pertes liées aux chambres bloquées inutilement


In [ ]:

#description de dataset

df.head()

In [ ]:
df.info()

**
Variables principales analysées :
- hotel : type d'hôtel (Resort ou City)
- lead_time : délai entre la réservation et l'arrivée
- arrival_date_month : mois d'arrivée
- stays_in_weekend_nights / stays_in_week_nights
- adults, children, babies
- meal, country, market_segment
- deposit_type : présence d'un dépôt (très important)
- customer_type
- adr : prix moyen par nuit
**

In [ ]:
print(df.columns)

print(df.dtypes)

print(df.shape)

In [ ]:
numeric_df = df.select_dtypes(include=['int64,float64'])
print(numeric_df)

cat_df = df.select_dtypes(include=['object'])
print(cat_df)


In [ ]:
print(df.describe())

In [ ]:
print(df.isnull().sum())

In [ ]:
import missingno as msno
msno.bar(df)

**L'analyse via isnull().sum() et la visualisation missingno confirment la présence de données manquantes sur 4 variables . Le point critique est la colonne company qui est presque totalement vide, suivie de la colonne agent. Les variables children et country ont des manques mineurs. **

In [ ]:


# Affichage des valeurs manquantes sous forme de heatmap
plt.figure(figsize=(12, 6))
sns.heatmap(df.isnull(), cbar=False, cmap="viridis")
plt.title("Carte des Valeurs Manquantes")
plt.show()

In [ ]:
#Suppression des colonnes reservation_status et reservation_status_date
df = df.drop(['reservation_status', 'reservation_status_date'], axis=1)

**Ces colonnes présentent un risque élevé de Data Leakage (fuite de données). Dans un projet de prédiction de l'annulation, le statut de la réservation est soit la cible elle-même, soit une variable directement corrélée à la réponse. Les inclure comme variables explicatives reviendrait à donner la "réponse" au modèle, rendant les performances artificiellement élevées mais inutilisables en conditions réelles.**

In [ ]:
print(df.isnull().sum())

In [ ]:
profile1 = ProfileReport(df)
profile1.to_notebook_iframe()

La variable children représente un nombre d'individus. Elle suit généralement une distribution asymétrique (beaucoup de réservations avec 0 ou 1 enfant, et très peu avec des nombres élevés). L'utilisation de la médiane est préférable à la moyenne car elle est robuste aux valeurs aberrantes (outliers). Elle permet de combler les 4 valeurs manquantes par une valeur centrale réaliste qui ne déforme pas la distribution globale.

In [ ]:
df['children'] = df['children'].fillna(df['children'].median())

In [ ]:
df['country'] = df['country'].fillna('Unknown')

Pour une variable catégorielle, on ne peut pas calculer de moyenne. Nous avons choisi de créer une catégorie spécifique 'Unknown'  plutôt que d'utiliser le mode (le pays le plus fréquent). Cela permet de préserver l'information liée à l'absence de donnée : le fait que le pays soit inconnu peut être, en soi, un signal statistique pour le modèle, plutôt que de faire croire à tort que ces clients viennent du pays le plus représenté.

In [ ]:
df['agent'] = df['agent'].fillna(0)
df['company'] = df['company'].fillna(0)

une valeur manquante signifie souvent qu'il n'y a ni agent, ni entreprise impliquée dans la réservation (réservation directe). L'utilisation de la valeur 0 sert ici de "placeholder" (indicateur de présence/absence). Cela permet de transformer l'absence d'intermédiaire en une information numérique claire que le modèle pourra interpréter comme une catégorie "Directe".

In [ ]:
print(df.isnull().sum())

In [ ]:
# Affichage des valeurs manquantes sous forme de heatmap
plt.figure(figsize=(12, 6))
sns.heatmap(df.isnull(), cbar=False, cmap="viridis")
plt.title("Carte des Valeurs Manquantes")
plt.show()

Après avoir traité les variables critiques (imputation des enfants, du pays et des agents), nous avons validé la qualité du dataset. Comme le montrent le tableau de bord et la heatmap, nous avons atteint un taux de complétude de 100%. Nous pouvons passer à l'étape suivante en toute confiance

In [ ]:
df = df[(df['adults'] + df['children'] + df['babies']) > 0]

df.shape

In [ ]:
# 2. Annulation selon le lead_time

plt.figure(figsize=(8, 5))
sns.barplot(data=df, x='hotel', y='is_canceled', hue='hotel', palette='Set2', legend=False)
plt.title("Taux d'annulation par type d'hôtel")
plt.xlabel("Type d'hôtel")
plt.ylabel("Taux d'annulation (entre 0 et 1)")

plt.show()

Le graphique montre une différence marquée entre les deux types d'établissements. Le City Hotel présente un taux d'annulation nettement plus élevé (environ 42%) par rapport au Resort Hotel (environ 28%)

In [ ]:
# --- GRAPHIQUE 2 : IMPACT DU LEAD TIME SUR L'ANNULATION ---

plt.figure(figsize=(8, 5))
sns.boxplot(data=df, x='is_canceled', y='lead_time', palette='Set1')
plt.title("Distribution du délai de réservation (Lead Time) selon l'annulation")
plt.xlabel("Réservation annulée ? (0 = Non, 1 = Oui)")
plt.ylabel("Nombre de jours d'avance (Lead Time)")

plt.show()

 Le boxplot compare la distribution du lead_time (nombre de jours entre la réservation et l'arrivée) selon que la réservation est annulée ou non
 Plus le délai de réservation est long, plus la probabilité d'annulation augmente. Les clients qui réservent très longtemps à l'avance ont tendance à changer leurs plans, augmentant ainsi le risque d'annulation pour l'hôtel

In [ ]:
# --- GRAPHIQUE 3 : IMPACT DU TYPE DE DÉPÔT SUR L'ANNULATION ---

plt.figure(figsize=(10, 6))


sns.countplot(data=df, x='deposit_type', hue='is_canceled', palette='Set2')

# Personnalisation simple
plt.title("Nombre de réservations selon le type de dépôt et l'annulation")
plt.xlabel("Type de dépôt (Deposit Type)")
plt.ylabel("Nombre de réservations")
plt.legend(title='Statut', labels=['Annulée', 'Confirmée'])

plt.show()

Le graphique montre une corrélation directe entre le type de dépôt et le statut de la réservation. Les réservations 'No Deposit' ont un taux d'annulation très élevé, tandis que les réservations 'Non Refund' sont presque systématiquement confirmées. La variable deposit_type est donc un facteur prédictif majeur pour notre modèle.

In [ ]:
plt.figure(figsize=(8, 4))
sns.boxplot(x=df['adr'], color='lightcoral')
plt.title("Détection des valeurs aberrantes sur l'ADR (Prix moyen)")
plt.show()

In [ ]:
Q1 = df['adr'].quantile(0.25)
Q3 = df['adr'].quantile(0.75)
IQR = Q3 - Q1
Borne_inf = Q1 - 1.5 * IQR
Borne_sup = Q3 + 1.5 * IQR


In [ ]:
print(f"Limite inférieure : {Borne_inf}")
print(f"Limite supérieure : {Borne_sup}")

# 4. Filtrer le dataset pour garder uniquement les valeurs dans les bornes
# Note : On vérifie aussi que l'ADR est supérieur à 0 pour éviter les prix aberrants de 0€
df = df[(df['adr'] >= 0) & (df['adr'] <= Borne_sup)]


In [ ]:
plt.figure(figsize=(8, 4))
sns.boxplot(x=df['adr'], color='lightcoral')
plt.title("Détection des valeurs aberrantes sur l'ADR (Prix moyen)")
plt.show()

In [ ]:
plt.figure(figsize=(8, 4))
sns.boxplot(x=df['lead_time'], color='lightcoral')
plt.title("Détection des valeurs aberrantes sur lead_time")
plt.show()

In [ ]:
Q1 = df['lead_time'].quantile(0.25)
Q3 = df['lead_time'].quantile(0.75)
IQR = Q3 - Q1
Borne_inf = Q1 - 1.5 * IQR
Borne_sup = Q3 + 1.5 * IQR

print(f"Limite inférieure : {Borne_inf}")
print(f"Limite supérieure : {Borne_sup}")

# 4. Filtrer le dataset pour garder uniquement les valeurs dans les bornes
# Note : On vérifie aussi que l'ADR est supérieur à 0 pour éviter les prix aberrants de 0€
df = df[(df['lead_time'] >= 0) & (df['lead_time'] <= Borne_sup)]


plt.figure(figsize=(8, 4))
sns.boxplot(x=df['lead_time'], color='lightcoral')
plt.title("Détection des valeurs aberrantes sur lead_time )")
plt.show()

In [ ]:
plt.figure(figsize=(8, 4))
sns.boxplot(x=df['stays_in_weekend_nights'], color='lightcoral')
plt.title("Détection des valeurs aberrantes sur stays_in_weekend_nights")
plt.show()

In [ ]:
Q1 = df['stays_in_weekend_nights'].quantile(0.25)
Q3 = df['stays_in_weekend_nights'].quantile(0.75)
IQR = Q3 - Q1
Borne_inf = Q1 - 1.5 * IQR
Borne_sup = Q3 + 1.5 * IQR

print(f"Limite inférieure : {Borne_inf}")
print(f"Limite supérieure : {Borne_sup}")

# 4. Filtrer le dataset pour garder uniquement les valeurs dans les bornes
# Note : On vérifie aussi que l'ADR est supérieur à 0 pour éviter les prix aberrants de 0€
df = df[(df['stays_in_weekend_nights'] >= 0) & (df['stays_in_weekend_nights'] <= Borne_sup)]


plt.figure(figsize=(8, 4))
sns.boxplot(x=df['stays_in_weekend_nights'], color='lightcoral')
plt.title("Détection des valeurs aberrantes sur stays_in_weekend_nights )")
plt.show()

In [ ]:
plt.figure(figsize=(8, 4))
sns.boxplot(x=df['adults'], color='lightcoral')
plt.title("Détection des valeurs aberrantes sur adults")
plt.show()

In [ ]:
Q1 = df['adults'].quantile(0.25)
Q3 = df['adults'].quantile(0.75)
IQR = Q3 - Q1
Borne_inf = Q1 - 1.5 * IQR
Borne_sup = Q3 + 1.5 * IQR

print(f"Limite inférieure : {Borne_inf}")
print(f"Limite supérieure : {Borne_sup}")

# 4. Filtrer le dataset pour garder uniquement les valeurs dans les bornes
# Note : On vérifie aussi que l'ADR est supérieur à 0 pour éviter les prix aberrants de 0€
df['adults'] = np.clip(df['adults'], Borne_inf, Borne_sup)


plt.figure(figsize=(8, 4))
sns.boxplot(x=df['adults'], color='lightcoral')
plt.title("Détection des valeurs aberrantes sur adults )")
plt.show()

In [ ]:
plt.figure(figsize=(8, 4))
sns.boxplot(x=df['children'], color='lightcoral')
plt.title("Détection des valeurs aberrantes sur children")
plt.show()

In [ ]:
Q1 = df['children'].quantile(0.25)
Q3 = df['children'].quantile(0.75)
IQR = Q3 - Q1
Borne_inf = Q1 - 1.5 * IQR
Borne_sup = Q3 + 1.5 * IQR

print(f"Limite inférieure : {Borne_inf}")
print(f"Limite supérieure : {Borne_sup}")

# 4. Filtrer le dataset pour garder uniquement les valeurs dans les bornes
# Note : On vérifie aussi que l'ADR est supérieur à 0 pour éviter les prix aberrants de 0€
df['children'] = df['children'].clip(lower=0, upper=5)

plt.figure(figsize=(8, 4))
sns.boxplot(x=df['children'], color='lightcoral')
plt.title("Détection des valeurs aberrantes sur children )")
plt.show()

In [ ]:
plt.figure(figsize=(8, 4))
sns.boxplot(x=df['babies'], color='lightcoral')
plt.title("Détection des valeurs aberrantes sur babies")
plt.show()

In [ ]:
Q1 = df['babies'].quantile(0.25)
Q3 = df['babies'].quantile(0.75)
IQR = Q3 - Q1
Borne_inf = Q1 - 1.5 * IQR
Borne_sup = Q3 + 1.5 * IQR

print(f"Limite inférieure : {Borne_inf}")
print(f"Limite supérieure : {Borne_sup}")

df['babies'] = df['babies'].clip(lower=0, upper=5)

plt.figure(figsize=(8, 4))
sns.boxplot(x=df['babies'], color='lightcoral')
plt.title("Détection des valeurs aberrantes sur babies )")
plt.show()

### Traitement des valeurs aberrantes

L’analyse visuelle à l’aide de boxplots a mis en évidence la présence de valeurs extrêmes pour les variables `adr` (prix moyen par nuit), `lead_time` (délai de réservation) ainsi que les durées de séjour (`stays_in_weekend_nights` et `stays_in_week_nights`).

Afin d’éviter que ces valeurs atypiques n’influencent excessivement les modèles tout en préservant la totalité des lignes du jeu de données, **nous avons appliqué une Winsorisation (capping)** sur ces variables.

